<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/jspace_workspace_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# J-space workspace readout: think X, say Y

The Jacobian lens reads an internal activation as the tokens it is disposed to make the model say, by transporting the
residual $h_\ell$ at layer $\ell$ into the final-layer basis with the average input-output Jacobian $\bar J_\ell$.
The steering demo (`ct_concept_steering_demo`) uses that space to *write*; this notebook uses it only to *read*.

The question is the one the Jacobian-lens paper asks of latent multi-step reasoning. When a prompt needs an
intermediate it never states, is that intermediate readable in the workspace before the answer is?

- **Say Y:** the answer the model gives (`racket`), read at the answer position layer by layer.
- **Think X:** the unstated bridge entity (`tennis`), which may surface at an earlier position rather than at the
  answer position, so it is read at every position and every fitted layer.

**Earlier Gemma 3 measurement.** It was chosen from the paper's two-hop set because the ordering is clear on it. Over
the 33 items of that set that gemma-3-1b-it answers correctly with single-token entities, the intermediate is readable
(lens top-5, any position) strictly before the answer on 24, the answer comes first or ties on 7, and the
intermediate is never readable on 2. One of the 7 (`the Amazon River ... Portuguese`, via `Brazil`) is the notebook's
second test parameterization, so the counter-case runs too. Rates with a specificity control are in
interpretune#425.
The default run now uses Qwen3.5-4B, whose public J-lens and J++ lens let section 6 compare both maps on the same
activations. The Gemma rate above is historical context, not a rate measured on Qwen. Set `MODEL_FAMILY="gemma3"`,
`MODEL_VARIANT="1b_it"`, `LENS_MODEL_ID="gemma-3-1b-it"` and `COMPARE_JPP=False` to repeat that model
(the public repository has no J++ lens for it).


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
MODEL_FAMILY = "qwen3_5"
MODEL_VARIANT = "4b"
LENS_MODEL_ID = "qwen3.5-4b"  # directory in neuronpedia/jacobian-lens
COMPARE_JPP = True
# A two-hop fact from the Jacobian-lens paper's own battery: the answer requires an intermediate the prompt never names.
PROMPT = "Fact: The piece of equipment used to strike the ball in the sport played at Wimbledon is a"
THINK_TOKEN = "tennis"  # the unstated intermediate ("think X")
SAY_TOKEN = "racket"  # the answer ("say Y")
RANK_DEPTH = 200  # a token outside the lens top-RANK_DEPTH is reported as not readable
READABLE_TOP_K = 5  # "readable" means inside the lens top-5

In [3]:
# @title Imports { display-mode: "form" }
import tempfile
from contextlib import ExitStack

import torch

import interpretune as it
import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.analysis.optools import resolve_jlens
from interpretune.analysis.points import component_map_for
from interpretune.analysis.ops.bundled.jlens.jlens_ops import default_jlens_cache_key
from interpretune.harness.sessions import experiment_session
from it_examples.utils.nb_ui_utils import display_jlens_readout_comparison

## 1: Session

The harness session machinery builds the model from a registered model spec. It is held open across cells with an
`ExitStack` and closed in the last cell.
Qwen3.5 uses the NNsight composition without circuit-tracer: readout requires no transcoders. Its registered
loading policy uses bf16. The Gemma parameterizations retain their circuit-tracer sessions.


In [4]:
_stack = ExitStack()
session, module, tokenizer = _stack.enter_context(
    experiment_session(
        tempfile.mkdtemp(), "jspace_workspace_demo", model_family=MODEL_FAMILY, model_variant=MODEL_VARIANT
    )
)
# the HF model under the nnsight wrapper (`_model`; `_module` on older nnsight releases)
hf = getattr(module.model, "_model", None) or getattr(module.model, "_module", None)
device = next(hf.parameters()).device
print(type(hf).__name__, "on", device)

[INFO] interpretune.utils.logging: Initializing NNsight LanguageModel with model: Qwen/Qwen3.5-4B


[INFO] interpretune.utils.logging: Qwen/Qwen3.5-4B is registered as multimodal; using NNsight VisionLanguageModel


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight LanguageModel initialized successfully


[INFO] interpretune.utils.logging: Preparing data: ExperimentDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: ExperimentDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


Qwen3_5ForConditionalGeneration on cuda:0


## 2: One forward pass, every block output captured

Each lens layer was fitted on its block's **output**, which is also the ops' default cache point
(`default_jlens_cache_key`). One forward pass with a hook on every decoder layer gives all of them.

In [5]:
def _single_token(text):
    ids = [i for i in tokenizer.encode(" " + text, add_special_tokens=False) if tokenizer.decode([i]).strip()]
    if len(ids) != 1:
        raise ValueError(f"{text!r} is {len(ids)} content tokens; each concept needs a single J-space row")
    return ids[0]


think_id, say_id = _single_token(THINK_TOKEN), _single_token(SAY_TOKEN)
input_ids = tokenizer((tokenizer.bos_token or "") + PROMPT, return_tensors="pt", add_special_tokens=False).input_ids
input_ids = input_ids.to(device)
prompt_tokens = [tokenizer.decode([i]) for i in input_ids[0].tolist()]

cmap = component_map_for(type(hf).__name__)
n_layers = hf.config.get_text_config().num_hidden_layers
layers = [hf.get_submodule(cmap.module_for("", idx)) for idx in range(n_layers)]
acts = {}
_hooks = [
    layer.register_forward_hook(
        lambda _m, _i, out, idx=idx: acts.__setitem__(idx, (out[0] if isinstance(out, tuple) else out).detach())
    )
    for idx, layer in enumerate(layers)
]
try:
    with torch.no_grad():
        logits = hf(input_ids).logits
finally:
    for _h in _hooks:
        _h.remove()
print("prompt:", PROMPT)
print("model answers:", repr(tokenizer.decode([int(logits[0, -1].argmax())])))

prompt: Fact: The piece of equipment used to strike the ball in the sport played at Wimbledon is a
model answers: ' racket'


## 3: What the workspace says at the answer position, layer by layer

`it.jlens_read` ranks the vocabulary by the lens readout. Each concept's rank comes from the top-`RANK_DEPTH` list
(0 is top-1).

In [6]:
def read(layer, positions):
    key = default_jlens_cache_key(layer)
    return it.jlens_read(
        module,
        AnalysisBatch(cache={key: acts[layer]}),
        None,
        0,
        jlens_layer=layer,
        jlens_model_id=LENS_MODEL_ID,
        jlens_positions=positions,
        jlens_top_k=RANK_DEPTH,
    )


def rank_in(ids_row, token_id):
    hits = (ids_row == token_id).nonzero()
    return int(hits[0]) if len(hits) else None


answer_pos = input_ids.shape[1] - 1
jlens_artifact = resolve_jlens(module, model_id=LENS_MODEL_ID, kind="jlens")
rows = []
for layer in jlens_artifact.source_layers:
    out = read(layer, [answer_pos])
    ids_row = torch.as_tensor(out["jlens_top_token_ids"])[0, 0]
    top = [tokenizer.decode([int(t)]) for t in ids_row[:5]]
    rows.append((layer, top, rank_in(ids_row, think_id), rank_in(ids_row, say_id)))


def _fmt(r):
    return "-" if r is None else str(r)


print(f"{'layer':>5} | {'top-5 at the answer position':<60} | {THINK_TOKEN} rank | {SAY_TOKEN} rank")
for layer, top, r_think, r_say in rows:
    print(f"{layer:>5} | {' '.join(repr(t) for t in top):<60} | {_fmt(r_think):>10} | {_fmt(r_say):>10}")

~/worktrees/jlr-957/src/interpretune/base/components/mixins.py:123: Analysis configuration has not been set.


layer | top-5 at the answer position                                 | tennis rank | racket rank
    0 | '.' ';' ',' ' ${' ':'                                        |          - |          -
    1 | ' â' '...' '.</' ' Â' ' *'                                   |          - |          -
    2 | '.</' ';' '.' '».' '.,'                                      |          - |          -
    3 | '.' ';' '.[' '.»' '.,'                                       |          - |          -
    4 | '.' ';' '.,' ',' ':'                                         |          - |          -
    5 | '.' ';' '.".' '.»' '...'                                     |          - |          -
    6 | '.' '.".' '.,' '.»' ';'                                      |          - |          -
    7 | '.' '.".' '.»' ':.' '».'                                     |          - |          -
    8 | '.' '.".' '.»' ':.' '.\\'                                    |          - |          -
    9 | '.' '.".' '.\\' ':.' '."'               

## 4: Where the unstated intermediate is readable

The answer position is where the model *says* Y, but the intermediate X does not have to be read there. This map
marks, for each position and fitted layer, the rank of `THINK_TOKEN` when it is inside the lens top-`READABLE_TOP_K`.

In [7]:
first_pos = int(tokenizer.bos_token_id is not None and int(input_ids[0, 0]) == tokenizer.bos_token_id)
positions = list(range(first_pos, input_ids.shape[1]))  # omit BOS only when the input has one
think_ranks = {}
for layer, *_ in rows:
    ids_all = torch.as_tensor(read(layer, positions)["jlens_top_token_ids"])[0]
    think_ranks[layer] = [rank_in(ids_all[p], think_id) for p in range(len(positions))]

label_w = max(len(repr(t)) for t in [prompt_tokens[p] for p in positions])
print(" " * (label_w + 1) + " ".join(f"{layer:>3}" for layer in think_ranks))
for p, tok in enumerate([prompt_tokens[p] for p in positions]):
    marks = []
    for layer in think_ranks:
        r = think_ranks[layer][p]
        marks.append(f"{r:>3}" if r is not None and r < READABLE_TOP_K else "  .")
    print(f"{repr(tok):>{label_w}} " + " ".join(marks))

readable = [
    (layer, p)
    for layer, ranks in think_ranks.items()
    for p, r in enumerate(ranks)
    if r is not None and r < READABLE_TOP_K
]
first_think = min((layer for layer, _ in readable), default=None)
first_say = min((layer for layer, _, _, r in rows if r is not None and r < READABLE_TOP_K), default=None)
print()
print(f"{THINK_TOKEN} first readable (any position): layer {first_think}")
print(f"{SAY_TOKEN} first readable (answer position): layer {first_say}")

               0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30
      'Fact'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
         ':'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
      ' The'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
    ' piece'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
       ' of'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
' equipment'   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .   .
     ' used'   .   .   .   .   .   .   . 

## 5: Think X versus say Y, side by side

The workspace at the layer and position where the intermediate is best read, against the answer position at the
last fitted layer, where the lens is closest to the model's own output.

In [8]:
if readable:
    best_layer, best_pos = min(readable, key=lambda lp: think_ranks[lp[0]][lp[1]])
    say_layer = rows[-1][0]

    def _top_pairs(out, k=10):
        ids = torch.as_tensor(out["jlens_top_token_ids"])[0, 0][:k]
        scores = torch.as_tensor(out["jlens_top_token_scores"])[0, 0][:k]
        return [(tokenizer.decode([int(t)]), float(s)) for t, s in zip(ids, scores)]

    think_top = _top_pairs(read(best_layer, [positions[best_pos]]))
    say_top = _top_pairs(read(say_layer, [answer_pos]))
    print(f"left: layer {best_layer} at {prompt_tokens[positions[best_pos]]!r}; right: layer {say_layer} at the answer")
    summary = display_jlens_readout_comparison(think_top, say_top, layer=best_layer, basis="jlens_folded")
else:
    print(f"{THINK_TOKEN} is not readable inside the top-{READABLE_TOP_K} at any position or layer for this prompt.")

left: layer 20 at ' ball'; right: layer 30 at the answer


#,Pre token,Pre,Post token,Post,Move
1,tennis,+10.14,racket,+26.37,new
2,baseball,+10.00,rac,+23.17,new
3,ball,+9.45,tennis,+22.83,-2
4,football,+9.42,Rac,+19.10,new
5,golf,+9.37,wooden,+18.69,new
6,"\""",+9.36,r,+18.30,new
7,soccer,+8.95,Tennis,+18.27,new
8,during,+8.90,ratt,+17.19,new
9,".\""",+8.76,rake,+16.95,new
10,volleyball,+8.71,lawn,+16.63,new


## 6: Read the same workspace through J-lens and J++

J++ is fitted for readout recall, while the J-lens averages causal Jacobians. Compare token ranks on the **same
prompt, activation and fitted layer**. Neither readout establishes that a token is a causal intermediate.

J++ defaults to a semantic-token mask (at least one letter or digit in the decoded token) and the final norm's RMS
divisor. The comparison applies that same policy explicitly to the J-lens, so a vocabulary filter cannot explain
a rank difference. It prints the policies returned by each op and the two artifact paths. The earlier J-lens sections
retain the J-lens defaults. Cross-lens raw scores are not compared.


In [9]:
if COMPARE_JPP:
    jpp_artifact = resolve_jlens(module, model_id=LENS_MODEL_ID, kind="jpp")
    common_layers = sorted(set(jlens_artifact.source_layers) & set(jpp_artifact.source_layers))
    if not common_layers:
        raise ValueError("The J-lens and J++ artifacts have no common fitted layers")
    print("J-lens:", jlens_artifact.path)
    print("J++:", jpp_artifact.path)
    print("layer | lens | intermediate rank | answer rank | top-5")
    comparison_rows = []
    for layer in common_layers:
        key = default_jlens_cache_key(layer)
        for kind in ("jlens", "jpp"):
            policy = {"jlens_readout_mask": "semantic", "jlens_include_rms_scale": True} if kind == "jlens" else {}
            out = it.jlens_read(
                module,
                AnalysisBatch(cache={key: acts[layer]}),
                None,
                0,
                jlens_layer=layer,
                jlens_model_id=LENS_MODEL_ID,
                jlens_kind=kind,
                jlens_positions=[answer_pos],
                jlens_top_k=RANK_DEPTH,
                **policy,
            )
            assert out["jlens_kind"] == kind
            assert out["jlens_readout_mask"] == "semantic" and out["jlens_include_rms_scale"] is True
            ids_row = torch.as_tensor(out["jlens_top_token_ids"])[0, 0]
            ranks = (rank_in(ids_row, think_id), rank_in(ids_row, say_id))
            comparison_rows.append((layer, kind, *ranks))
            top = [tokenizer.decode([int(t)]) for t in ids_row[:5]]
            print(layer, kind, *(_fmt(r) for r in ranks), repr(top), sep=" | ")
    print("comparison policy: semantic tokens, RMS divisor; J++ defaults verified")
else:
    print("J++ comparison disabled for this model parameterization")

J-lens: qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens.pt
J++: qwen3.5-4b/jpp/Salesforce-wikitext/Qwen3.5-4B_jpp_lens.pt
layer | lens | intermediate rank | answer rank | top-5


0 | jlens | - | - | ['<|endoftext|>', 'â', ' stern', ' â', '东山']


0 | jpp | - | - | [' a', ' the', ' heavily', ' unofficial', ' part']


1 | jlens | - | - | [' â', ' Â', '<|object_ref_start|>', 'â', '<|endoftext|>']


1 | jpp | - | - | [' but', ' not', ' mostly', ' at', ' both']


2 | jlens | - | - | ['s', '<|endoftext|>', ' Â', ' later', 'Â']


... (53 outputs left out)

29 | jlens | 2 | 0 | [' racket', ' rac', ' tennis', ' wooden', ' Tennis']


29 | jpp | 2 | 1 | [' rac', ' racket', ' tennis', ' Rac', ' wooden']


30 | jlens | 2 | 0 | [' racket', ' rac', ' tennis', ' Rac', ' wooden']


30 | jpp | 4 | 0 | [' racket', ' rac', ' r', ' Rac', ' tennis']
comparison policy: semantic tokens, RMS divisor; J++ defaults verified


## Reading the result

The verdict below is computed from this run rather than written in advance. When the intermediate is readable at an
earlier layer than the answer, that is the latent-reasoning signature the Jacobian-lens paper describes, read here
through interpretune's `jlens_read` on a published lens. When it is not, the prompt is one of the cases where the
answer surfaces first or at the same depth.

The steering demo (`ct_concept_steering_demo`, sections 4b to 4d) uses the same space to write rather than read.

In [10]:
if first_think is None:
    verdict = f"{THINK_TOKEN} never became readable; this prompt does not show the ordering"
elif first_say is None or first_think < first_say:
    verdict = f"{THINK_TOKEN} was readable at layer {first_think}, before {SAY_TOKEN} (layer {first_say})"
else:
    verdict = f"{SAY_TOKEN} was readable at layer {first_say}, no later than {THINK_TOKEN} (layer {first_think})"
print(verdict)
_stack.close()

tennis was readable at layer 8, before racket (layer 24)
